# A Boundary-Layer Mechanism for One-Third Scaling in Online Softmax Classification

This notebook provides the numerical implementation and empirical validation for the boundary-layer mechanism described in the manuscript. We investigate how the mismatch between smooth surrogate losses (like softmax cross-entropy) and discrete hard labels produces predictable power-law learning curves in an online teacher-student framework.

The experiments are organized into seven parts, covering the core theory, controlled departures, and empirical image and language-model settings. **Note:** Some Gaussian hyperparameters are scaled down for illustration, while the minibatch sweep, mechanism, MNIST, CIFAR-5M, and Pythia experiments can require substantial compute. Run the notebook from the directory containing this file so that local imports and output paths resolve correctly.


## 1. The Core Mechanism: I.I.D. Gaussian Data

We begin with the standard thermodynamic-limit setting using isotropic Gaussian inputs. This simplifies the macroscopic dynamics, allowing us to cleanly isolate the $O(D^{-1})$ boundary-layer effects where the network's predictions remain uncertain.

The following code evaluates the generalization error $\epsilon_g$, the test loss $\mathcal{L}_g$ the centered overlap $D$, and the residual variance $\Delta$ against the macroscopic training time $\alpha$. This confirms the main $\alpha^{-1/3}$ power law at fixed learning rates as well as the predictions for learning rate schedules, when using different hyperparameters.

In [ ]:
from util_iid_gaussian import *

run_parameter_sweep(
        eta_list=[3.0, 0.1, 0.003], 
        gamma_list=[0.0], 
        seeds=[0,1],
        num_test=10000,
        data_dir="data_K_class",
        fig_dir="figure_K_class",
        N=100, 
        K=3, 
        alpha_max=10000, 
        alpha_min=200.0, 
        alpha_0=200.0,
        d=200, 
        spacing='log',
        first_step=1,
        num_saves=200,
        ylim_D=2e-2
    )

## 2. Controlled Departures: Correlated Inputs

Real-world datasets naturally exhibit structured covariance. To test the robustness of the boundary-layer mechanism, we introduce a controlled power-law covariance spectrum $\langle \xi_i^2 \rangle \sim i^{-\beta}$ to the input data.

By sweeping the anisotropy parameter $\beta$, we can observe how structured data impacts the length and shape of the transient learning phase, while verifying if the late-time boundary-layer decay remains consistent.

In [ ]:
from util_correlated_gaussian import *

fixed_params = {
    'N': 100, 'K': 3, 'alpha_max': 10000, 'beta':1.0,
    'a': 10.0, 'num_saves': 100, "num_test": 10000, "eta":0.5, 'gamma':0.0,
}

param_grid = {
    'eta': [0.5], 
    'beta': [0.0,0.5,1.0,1.5], 
    'seed': [0,1]
}

grid_search(param_grid, fixed_params)


plot_sweep(sweep_param='beta', param_values=param_grid['beta'], seeds=param_grid['seed'], fixed_params=fixed_params)


## 3. Non-Gaussian Test Bed: Pretrained ViT Features

Finally, we relax the Gaussian assumption entirely by training on representations extracted from a pretrained Vision Transformer (ViT-B/16) on the CIFAR-5M dataset. 

We apply ZCA whitening to remove the leading covariance structure, leaving a non-Gaussian feature distribution. We report both real-label and teacher-label training; the teacher-label variant bypasses early irreducible Bayes error floors so that the late-time scaling regime remains visible for evaluation.

*(Requires CIFAR-5M dataset to be downloaded locally prior to execution.)*

In [ ]:
%run cifar5m_vit_features.py

In [ ]:
%run train_cifar5m.py --data_file cifar5m_vit_features.pt --lr 0.0003 --seeds 0 1 2 3 --num_examples 1800000 --test_split 0.01 --use_whitening --bs 1


In [ ]:
%run plot_cifar5m.py

## 4. Minibatch-Size Effects

To test how gradient averaging changes the dynamics, we repeat the i.i.d. Gaussian experiment at a fixed learning rate for several minibatch sizes. The supplied sweep uses the manuscript-scale horizon and stores reusable results in the same Gaussian data and figure directories.


In [ ]:
from util_batch_size import run_parameter_sweep as run_batch_size_sweep

run_batch_size_sweep(
        batch_size_list=[1, 16, 128],
        eta=0.1,
        seeds=[0,1,2,3],
        num_test=10000,
        data_dir="data_K_class",
        fig_dir="figure_K_class",
        N=200,
        K=3,
        alpha_max=100000,
        alpha_min=200.0,
        d=200,
        spacing='log',
        first_step=1,
        num_saves=200,
        ylim_D=2e-2,
        plot_kwargs={"figsize": [2.45,1.5], "figsize_small": [1.45,1.5],
                     "inset_dims": [0.20,0.15,0.30,0.38],
                     "ax_dims": [0.17,0.22,0.80,0.74],
                     "ax_dims_small": [0.32,0.22,0.65,0.74]}
    )


## 5. Direct Boundary-Layer Diagnostics

This experiment conditions the error and alignment gradient on the teacher gap as alignment grows, directly resolving the shrinking boundary layer. Its Monte Carlo evaluation is intentionally large and caches each completed seed under `data/mechanism/`.


In [ ]:
from util_mechanism import run_mechanism

params = {
    'N': 200,
    'K': 3,
    'eta': 0.5,
    'alphas': (100, 150, 200, 300, 400, 600, 900, 1300, 1800, 2600, 3800, 5400, 7800, 11300, 16200, 23400, 33600, 48300, 69500, 100000),
    'density_alphas': (100, 3800, 100000),
    'seeds': (0,),
    'n_eval': 20_000_000,
    'eval_batch': 200_000,
    'error_x_max': 0.6,
    'gradient_x_max': 2.7,
}

results, figures, axes = run_mechanism(**params)


## 6. MNIST Experiment

The first run downloads MNIST and trains six seeds. Later runs reuse `data/mnist_histories.npz`; the two manuscript plots are saved in `figures/`.


In [ ]:
from util_mnist import run_mnist

error_figure, loss_figure = run_mnist()


## 7. Pythia Language-Model Checkpoints

Before using the plotting cell, generate all three model results from a terminal:

```bash
python evaluate_pythia.py 410m
python evaluate_pythia.py 2.8b
python evaluate_pythia.py 12b
```


In [ ]:
from plot_pythia import plot_pythia

raw_figure, baseline_removed_figure = plot_pythia()
